# U1-01 · Python, NumPy, SciPy, NetworkX y Matplotlib

*Programación de Redes Neuronales - Machine Learning (IFCD48) · Unidad 1, apartados 1.1 y 1.2*

Ejecuta cada celda con **Mayús + Intro**. Todo funciona en el navegador, sin instalar nada.

## 1. Control del flujo y errores

In [ ]:
import csv

texto = """modelo,acierto,segundos
regresión logística,0.9262,43
red 784-128-10,0.9741,44
árbol de decisión,pendiente,
red 784-10,0.9269,38
"""
with open("experimentos.csv", "w", encoding="utf-8") as f:
    f.write(texto)

resultados = {}
with open("experimentos.csv", encoding="utf-8") as f:
    for fila in csv.DictReader(f):
        try:
            acierto = float(fila["acierto"])
            segundos = int(fila["segundos"])
        except ValueError:                  # dato vacío o erróneo
            print("fila incompleta:", fila["modelo"])
            continue
        resultados[fila["modelo"]] = (acierto, segundos)

mejor = max(resultados, key=lambda m: resultados[m][0])
print("mejor:", mejor, resultados[mejor])
for m, (a, s) in sorted(resultados.items(),
                        key=lambda x: -x[1][0]):
    marca = "rápido" if s < 40 else "lento"
    print(f"{m:<20} {a:.1%} {s:>3} s {marca}")
print(type(resultados), type(mejor), 7 / 2, 7 // 2, 2 ** 10)

## 2. Clases: el patrón fit y transform

In [ ]:
import numpy as np

class Estandarizador:
    """Resta la media y divide por la desviación de cada columna."""
    def fit(self, X):
        self.centro_ = X.mean(axis=0)
        self.escala_ = X.std(axis=0)
        return self                       # permite encadenar
    def transform(self, X):
        return (X - self.centro_) / self.escala_
    def fit_transform(self, X):
        return self.fit(X).transform(X)

class EstandarizadorRobusto(Estandarizador):
    """Igual, pero con la mediana y el rango intercuartílico."""
    def fit(self, X):
        q1, q2, q3 = np.percentile(X, [25, 50, 75], axis=0)
        self.centro_, self.escala_ = q2, q3 - q1
        return self

rng = np.random.default_rng(0)
X = rng.normal(50, 10, size=(1000, 3))
X[0] = 10_000                             # un valor atípico
for clase in (Estandarizador, EstandarizadorRobusto):
    Z = clase().fit_transform(X)
    print(f"{clase.__name__:<22} mediana de |z|:",
          np.median(np.abs(Z), axis=0).round(2))

from sklearn.preprocessing import StandardScaler
iguales = np.allclose(Estandarizador().fit_transform(X),
                      StandardScaler().fit_transform(X))
print("igual que scikit-learn:", iguales)
print(isinstance(EstandarizadorRobusto(), Estandarizador))

## 3. Ficheros y formatos

Se usan las 1000 primeras imágenes de MNIST, incluidas junto a este cuaderno; por eso algunas cifras difieren de las del libro, calculadas con las 60 000.

> Datos: MNIST, de Y. LeCun, C. Cortes y C. J. C. Burges, con licencia Creative Commons Reconocimiento-CompartirIgual 3.0.

In [ ]:
import os, json, pickle, time
import numpy as np

datos = np.load("mnist_muestra.npz")        # las 1000 primeras de MNIST
x_ent, y_ent = datos["x"], datos["y"]

def guarda(nombre, funcion):
    t = time.perf_counter(); funcion(nombre)
    t = time.perf_counter() - t
    print(f"{nombre:<16} {os.path.getsize(nombre) / 1e6:6.1f} MB"
          f" {t:5.2f} s")

guarda("mnist.npy", lambda f: np.save(f, x_ent))
guarda("mnist_c.npz", lambda f: np.savez_compressed(
    f, x=x_ent, y=y_ent))
guarda("mnist_1000.csv", lambda f: np.savetxt(
    f, x_ent[:1000].reshape(1000, -1), fmt="%d", delimiter=","))
guarda("mnist.pkl", lambda f: pickle.dump(
    x_ent, open(f, "wb")))
info = {"imagenes": len(x_ent), "forma": list(x_ent.shape[1:]),
        "clases": sorted(set(y_ent.tolist()))}
guarda("info.json", lambda f: json.dump(info, open(f, "w")))

d = np.load("mnist_c.npz")
print("recuperado igual:", np.array_equal(d["x"], x_ent))
print(json.load(open("info.json")))

## 4. Resultados reproducibles

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

a = np.random.default_rng(42).integers(0, 100, 5)
b = np.random.default_rng(42).integers(0, 100, 5)
c = np.random.default_rng().integers(0, 100, 5)   # sin semilla
print(a, b, "iguales:", np.array_equal(a, b))
print(c, "(cambia en cada ejecución)")

X, y = np.arange(20).reshape(10, 2), np.arange(10)
for semilla in (0, 0, 1):
    _, X_pru, _, y_pru = train_test_split(
        X, y, test_size=0.3, random_state=semilla)
    print("semilla", semilla, "-> prueba:", y_pru)

## 5. Pensar en vectores

In [ ]:
import time
import numpy as np

n = 1_000_000
lista = list(range(n))
vector = np.arange(n, dtype=np.float64)

t0 = time.perf_counter()
total = 0.0
for x in lista:                 # bucle de Python: elemento a elemento
    total += x * x
t_bucle = time.perf_counter() - t0

t0 = time.perf_counter()
total_np = np.dot(vector, vector)  # una sola operación vectorial
t_numpy = time.perf_counter() - t0

print(f"bucle: {t_bucle * 1000:.1f} ms, "
      f"NumPy: {t_numpy * 1000:.2f} ms")
print(f"{t_bucle / t_numpy:.0f} veces más rápido")
print(f"iguales: {total == total_np}; diferencia relativa: "
      f"{abs(total - total_np) / total:.1e}")
print("memoria:", vector.nbytes // 1024, "KB en float64,",
      vector.astype(np.float32).nbytes // 1024, "KB en float32")

**Prueba.** Cambia `n` a 10 millones. ¿Crece la ventaja de NumPy?

## 6. Una capa de red con NumPy

In [ ]:
import numpy as np

datos = np.load("mnist_muestra.npz")        # las 1000 primeras de MNIST
x_ent, y_ent = datos["x"], datos["y"]
lote = x_ent[:32]                         # un lote de 32 imágenes
print("lote:", lote.shape, lote.dtype)

X = lote.reshape(32, 784).astype(np.float32) / 255  # 32 vectores
print("imagen 0, píxeles 400 a 405:", X[0, 400:406].round(2))
print("imágenes que son un 7:", np.flatnonzero(y_ent[:32] == 7))

# una capa de red = producto de matrices + sesgo + activación
rng = np.random.default_rng(0)
W = rng.normal(0, 0.05, (784, 128)).astype(np.float32)
b = np.zeros(128, dtype=np.float32)
Z = X @ W + b                    # (32, 784) @ (784, 128) -> (32, 128)
H = np.maximum(Z, 0)             # ReLU: deja pasar solo lo positivo
print("salida de la capa:", H.shape,
      f"{(H > 0).mean():.0%} de las neuronas activas")

## 7. Softmax estable

In [ ]:
import numpy as np

def softmax_ingenua(z):
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def softmax(z):                     # estable: resta el máximo
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

z = np.array([[2.0, 1.0, 0.1],
              [1000.0, 999.0, 998.1]])   # misma diferencia
with np.errstate(over="ignore", invalid="ignore"):
    print("ingenua:", softmax_ingenua(z).round(3))
print("estable:", softmax(z).round(3))
print("suma por fila:", softmax(z).sum(axis=1))
print("clase elegida:", softmax(z).argmax(axis=1))
with np.errstate(over="ignore"):
    print("exp(709) =", np.exp(709.0), " exp(710) =", np.exp(710.0))

## 8. Valores singulares y compresión

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x_ent = np.load("mnist_muestra.npz")["x"]   # las 1000 primeras de MNIST
X = x_ent.reshape(len(x_ent), -1) / 255.0
media = X.mean(axis=0)
U, s, Vt = np.linalg.svd(X - media, full_matrices=False)
var = s**2 / np.sum(s**2)
for k in (10, 50, 100, 200):
    print(f"k = {k:3d}: {var[:k].sum():.1%} de la varianza")

def reconstruye(i, k):           # imagen i con k componentes
    return media + (U[i, :k] * s[:k]) @ Vt[:k]

ks = (5, 20, 50, 784)
fig, ejes = plt.subplots(1, len(ks), figsize=(4.4, 1.2))
for eje, k in zip(ejes, ks):
    r = reconstruye(0, k)
    eje.imshow(r.reshape(28, 28), vmin=0, vmax=1)
    eje.axis("off")
    eje.set_title(f"k = {k}", loc="center")
    print(f"k = {k:3d}: error medio por píxel "
          f"{np.abs(r - X[0]).mean():.3f}")
plt.show()  # fig.savefig("svd.pdf")

## 9. Aprender es optimizar

In [ ]:
import numpy as np
from scipy.optimize import minimize

rng = np.random.default_rng(0)
x = rng.uniform(0, 10, 50)
y = 2 * x + 1 + rng.normal(0, 1, 50)        # recta real: 2x + 1

def error(p):                               # error cuadrático medio
    w, b = p
    return np.mean((w * x + b - y) ** 2)

# 1) un optimizador de SciPy busca los parámetros
r = minimize(error, x0=[0.0, 0.0])
print(f"SciPy: w = {r.x[0]:.3f}, b = {r.x[1]:.3f}, "
      f"error = {r.fun:.3f}, {r.nit} iteraciones")

# 2) lo mismo, a mano, con descenso de gradiente
w, b, eta = 0.0, 0.0, 0.01
for paso in range(5000):
    e = w * x + b - y
    w -= eta * 2 * np.mean(e * x)           # derivada respecto a w
    b -= eta * 2 * np.mean(e)               # derivada respecto a b
print(f"gradiente: w = {w:.3f}, b = {b:.3f}, "
      f"error = {error([w, b]):.3f}")

# 3) solución exacta por mínimos cuadrados
A = np.column_stack([x, np.ones_like(x)])
print("exacta:   w = {:.3f}, b = {:.3f}".format(
    *np.linalg.lstsq(A, y, rcond=None)[0]))

**Prueba.** En el descenso de gradiente, cambia `eta` a 0.03 y a 0.001. ¿Qué ocurre con 0.03? ¿Y cuántos pasos harían falta con 0.001?

## 10. Intervalos de confianza para un acierto

In [ ]:
from scipy import stats

n = 10_000                                # imágenes de prueba
for nombre, aciertos in (("red sin capa oculta", 9269),
                         ("red con capa oculta", 9741)):
    ic = stats.binomtest(aciertos, n).proportion_ci(0.95)
    print(f"{nombre:<20} {aciertos / n:.2%}  IC 95 %: "
          f"{ic.low:.2%} a {ic.high:.2%}")

# precisión del acierto según el tamaño de la prueba
p = 0.974
for n in (1000, 10_000, 100_000):
    ic = stats.binomtest(round(p * n), n).proportion_ci(0.95)
    print(f"n = {n:>6}: ancho del intervalo "
          f"{(ic.high - ic.low) * 100:.2f} puntos")

## 11. Transformar imágenes con SciPy

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage

datos = np.load("mnist_muestra.npz")        # las 1000 primeras de MNIST
x_ent, y_ent = datos["x"], datos["y"]
img = x_ent[0] / 255.0                         # un 5
variantes = {
    "original": img,
    "desplazada": ndimage.shift(img, (0, 3)),
    "girada 20°": ndimage.rotate(img, 20, reshape=False),
    "suavizada": ndimage.gaussian_filter(img, 1),
}
fig, ejes = plt.subplots(1, 4, figsize=(4.4, 1.2))
for eje, (nombre, v) in zip(ejes, variantes.items()):
    r = np.corrcoef(img.ravel(), v.ravel())[0, 1]
    print(f"{nombre:<11} correlación con la original: {r:.2f}")
    eje.imshow(v, vmin=0, vmax=1); eje.axis("off")
    eje.set_title(nombre, loc="center")
plt.show()  # fig.savefig("ndimage.pdf")

## 12. Una red neuronal es un grafo

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

def red_en_capas(capas):
    """Grafo dirigido: cada neurona se une a todas las de la capa
    siguiente."""
    g = nx.DiGraph()
    for c, n in enumerate(capas):
        g.add_nodes_from(((c, i) for i in range(n)), capa=c)
    for c in range(len(capas) - 1):
        g.add_edges_from(((c, i), (c + 1, j))
                         for i in range(capas[c])
                         for j in range(capas[c + 1]))
    return g

g = red_en_capas([4, 5, 3])
print(g.number_of_nodes(), "neuronas,", g.number_of_edges(),
      "conexiones")
pos = nx.multipartite_layout(g, subset_key="capa")
nx.draw(g, pos, node_size=120, node_color="#1F5FA8",
        edge_color="#9AA5B1", arrowsize=6)
plt.savefig("grafo_red.pdf")

grande = red_en_capas([784, 128, 10])      # la red del apartado 1.10
pesos = grande.number_of_edges()
sesgos = 128 + 10                # uno por neurona oculta o de salida
print(f"red 784-128-10: {pesos:,} pesos + {sesgos} sesgos "
      f"= {pesos + sesgos:,} parámetros")
plt.show()

## 13. Una red social: el club de kárate

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from networkx.algorithms import community

G = nx.karate_club_graph()
print(G.number_of_nodes(), "socios,", G.number_of_edges(),
      "relaciones")
grado = nx.degree_centrality(G)
inter = nx.betweenness_centrality(G)
for n in sorted(grado, key=grado.get, reverse=True)[:3]:
    print(f"socio {n:2d}: {G.degree(n)} amigos, intermediación "
          f"{inter[n]:.2f}, club tras la escisión: "
          f"{G.nodes[n]['club']}")

grupos = community.greedy_modularity_communities(G)
print("comunidades:", [len(g) for g in grupos])
# cada comunidad se asigna al club mayoritario de sus socios
acierto = 0
for g in grupos:
    clubes = [G.nodes[n]["club"] for n in g]
    acierto += max(clubes.count(c) for c in set(clubes))
print(f"socios en el club que predice su comunidad: {acierto}/34")

color = {n: i for i, g in enumerate(grupos) for n in g}
plt.figure(figsize=(4.4, 2.6))
nx.draw(G, nx.spring_layout(G, seed=3), node_size=60,
        node_color=[color[n] for n in G], cmap="tab10",
        vmin=0, vmax=9, width=0.4, edge_color="0.6")
plt.show()  # plt.savefig("karate.pdf")

## 14. Anatomía de una figura

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

datos = np.load("mnist_muestra.npz")        # las 1000 primeras de MNIST
x_ent, y_ent = datos["x"], datos["y"]
tinta = (x_ent > 0).sum(axis=(1, 2))          # píxeles con tinta
media = [tinta[y_ent == d].mean() for d in range(10)]
print("tinta media por dígito:", np.round(media).astype(int))

fig, (a, b) = plt.subplots(1, 2, figsize=(4.4, 1.8),
                           layout="constrained")
for d in (1, 0):
    a.hist(tinta[y_ent == d], bins=40, alpha=0.7, label=str(d))
a.set_xlabel("píxeles con tinta"); a.set_ylabel("imágenes")
a.legend(title="dígito"); a.set_title("A. Distribución")
b.bar(range(10), media)
b.set_xticks(range(10)); b.set_xlabel("dígito")
b.set_title("B. Tinta media")
plt.show()  # fig.savefig("anatomia.pdf")

**Ejercicio.** Calcula con `red_en_capas` los parámetros de una red 784-256-128-10.

> Harris, C. R. et al. (2020). Array programming with NumPy. *Nature*, 585, 357–362. https://doi.org/10.1038/s41586-020-2649-2
>
> Virtanen, P. et al. (2020). SciPy 1.0. *Nature Methods*, 17, 261–272. https://doi.org/10.1038/s41592-019-0686-2
>
> Zachary, W. W. (1977). An information flow model for conflict and fission in small groups. *Journal of Anthropological Research*, 33(4), 452–473. https://doi.org/10.1086/jar.33.4.3629752